# DA6401 Assignment 3 — Transformer NMT (De→En)


In [ ]:
import zipfile, shutil, os, glob

zip_path = glob.glob('/kaggle/input/*/*.zip')[0]
print('Found zip:', zip_path)

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('/kaggle/working/')

for f in glob.glob('/kaggle/working/a3/*'):
    shutil.copy(f, '/kaggle/working/')

print('Files in working dir:', os.listdir('/kaggle/working/'))

In [ ]:
!pip install -q wandb evaluate sacrebleu gdown spacy datasets
!python -m spacy download de_core_news_sm -q
!python -m spacy download en_core_web_sm -q

In [ ]:
import wandb
wandb.login(key='YOUR_WANDB_API_KEY')

## Step 1: Build Vocabularies & Dataloaders


In [ ]:
import sys
sys.path.insert(0, '/kaggle/working')

import torch
from torch.utils.data import DataLoader, Dataset
from functools import partial
from dataset import Multi30kDataset

train_ds = Multi30kDataset('train')
val_ds   = Multi30kDataset('validation')
test_ds  = Multi30kDataset('test')

train_ds.build_vocab(train_ds.data, min_freq=2)
for ds in [val_ds, test_ds]:
    ds.src_vocab = train_ds.src_vocab
    ds.src_itos  = train_ds.src_itos
    ds.tgt_vocab = train_ds.tgt_vocab
    ds.tgt_itos  = train_ds.tgt_itos

train_pairs = train_ds.process_data()
val_pairs   = val_ds.process_data()
test_pairs  = test_ds.process_data()

PAD_IDX = train_ds.tgt_vocab['<pad>']
SOS_IDX = train_ds.tgt_vocab['<sos>']
EOS_IDX = train_ds.tgt_vocab['<eos>']

print(f'src_vocab={len(train_ds.src_vocab)}, tgt_vocab={len(train_ds.tgt_vocab)}')
print(f'train={len(train_pairs)}, val={len(val_pairs)}, test={len(test_pairs)}')

In [ ]:
class TokenDataset(Dataset):
    def __init__(self, pairs): self.pairs = pairs
    def __len__(self): return len(self.pairs)
    def __getitem__(self, i): return self.pairs[i]

def collate_fn(batch, pad_idx=1):
    srcs, tgts = zip(*batch)
    sm = max(len(s) for s in srcs)
    tm = max(len(t) for t in tgts)
    sp = torch.tensor([s + [pad_idx]*(sm-len(s)) for s in srcs], dtype=torch.long)
    tp = torch.tensor([t + [pad_idx]*(tm-len(t)) for t in tgts], dtype=torch.long)
    return sp, tp

_cf = partial(collate_fn, pad_idx=PAD_IDX)
train_loader = DataLoader(TokenDataset(train_pairs), batch_size=128, shuffle=True,  collate_fn=_cf)
val_loader   = DataLoader(TokenDataset(val_pairs),   batch_size=128, shuffle=False, collate_fn=_cf)
test_loader  = DataLoader(TokenDataset(test_pairs),  batch_size=64,  shuffle=False, collate_fn=_cf)

print('Dataloaders ready.')

## Step 2: Save Vocab to Disk


In [ ]:
torch.save({
    'src_vocab': train_ds.src_vocab,
    'tgt_vocab': train_ds.tgt_vocab,
    'src_itos':  train_ds.src_itos,
    'tgt_itos':  train_ds.tgt_itos,
}, '/kaggle/working/vocabs.pt')
print('Saved vocabs.pt')

## Step 3: Main Training Run (Noam + Label Smoothing)


In [ ]:
import torch.nn as nn
from model import Transformer
from lr_scheduler import NoamScheduler
from train import LabelSmoothingLoss, run_epoch, evaluate_bleu, save_checkpoint

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

CFG = dict(
    d_model=256, N=3, num_heads=8, d_ff=512,
    dropout=0.1, warmup_steps=4000,
    num_epochs=20, label_smoothing=0.1
)

run = wandb.init(project='da6401-a3', name='main-noam-ls01', config=CFG)

model = Transformer(
    src_vocab_size=len(train_ds.src_vocab),
    tgt_vocab_size=len(train_ds.tgt_vocab),
    d_model=CFG['d_model'],
    N=CFG['N'],
    num_heads=CFG['num_heads'],
    d_ff=CFG['d_ff'],
    dropout=CFG['dropout'],
    pad_idx=PAD_IDX,
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
scheduler = NoamScheduler(optimizer, d_model=CFG['d_model'], warmup_steps=CFG['warmup_steps'])
loss_fn   = LabelSmoothingLoss(len(train_ds.tgt_vocab), PAD_IDX, CFG['label_smoothing'])

best_val = float('inf')
for epoch in range(CFG['num_epochs']):
    tr = run_epoch(train_loader, model, loss_fn, optimizer, scheduler, epoch, True,  device)
    vl = run_epoch(val_loader,   model, loss_fn, None,      None,      epoch, False, device)
    wandb.log({'train_loss': tr, 'val_loss': vl, 'epoch': epoch,
               'lr': optimizer.param_groups[0]['lr']})
    print(f'Epoch {epoch+1:02d}  train={tr:.4f}  val={vl:.4f}')
    if vl < best_val:
        best_val = vl
        save_checkpoint(model, optimizer, scheduler, epoch, '/kaggle/working/best_checkpoint.pt')
        print('  -> checkpoint saved')

run.finish()
print('Training complete.')

## Step 4: BLEU on Test Set


In [ ]:
ckpt = torch.load('/kaggle/working/best_checkpoint.pt', map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

class SimpleVocab:
    def __init__(self, stoi, itos_d):
        self.stoi = stoi
        self.itos = itos_d

tgt_vocab_obj = SimpleVocab(train_ds.tgt_vocab, train_ds.tgt_itos)
bleu = evaluate_bleu(model, test_loader, tgt_vocab_obj, device)
print(f'Test BLEU: {bleu:.2f}')

---
## W&B Experiments
### Experiment 1: Noam vs Fixed LR


In [ ]:
EPOCHS_ABL = 10
base_cfg = dict(d_model=256, N=3, num_heads=8, d_ff=512, dropout=0.1)

def build_model(cfg, src_vsz, tgt_vsz, pad_idx, device):
    m = Transformer(
        src_vocab_size=src_vsz, tgt_vocab_size=tgt_vsz,
        d_model=cfg['d_model'], N=cfg['N'],
        num_heads=cfg['num_heads'], d_ff=cfg['d_ff'],
        dropout=cfg['dropout'], pad_idx=pad_idx
    ).to(device)
    return m

for exp_name, use_noam in [('fixed_lr', False), ('noam_lr', True)]:
    run = wandb.init(project='da6401-a3', name=f'exp1-{exp_name}',
                     config={**base_cfg, 'use_noam': use_noam})
    m = build_model(base_cfg, len(train_ds.src_vocab), len(train_ds.tgt_vocab), PAD_IDX, device)
    if use_noam:
        opt = torch.optim.Adam(m.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
        sched = NoamScheduler(opt, d_model=base_cfg['d_model'], warmup_steps=4000)
    else:
        opt = torch.optim.Adam(m.parameters(), lr=1e-4)
        sched = None
    lf = LabelSmoothingLoss(len(train_ds.tgt_vocab), PAD_IDX, 0.1)
    for ep in range(EPOCHS_ABL):
        tr = run_epoch(train_loader, m, lf, opt, sched, ep, True,  device)
        vl = run_epoch(val_loader,   m, lf, None, None,   ep, False, device)
        wandb.log({'train_loss': tr, 'val_loss': vl, 'epoch': ep})
        print(f'[{exp_name}] Epoch {ep+1}  train={tr:.4f}  val={vl:.4f}')
    run.finish()
    print(f'{exp_name} done\n')

### Experiment 2: Ablation — Scaling Factor 1/sqrt(dk)


In [ ]:
import math
import torch.nn.functional as F
import model as model_module
from model import make_src_mask, make_tgt_mask

_orig_sdpa = model_module.scaled_dot_product_attention

def sdpa_no_scale(Q, K, V, mask=None):
    scores = torch.matmul(Q, K.transpose(-2, -1))
    if mask is not None:
        scores = scores.masked_fill(mask, float('-inf'))
    attn_w = F.softmax(scores, dim=-1)
    return torch.matmul(attn_w, V), attn_w

for exp_name, use_scale in [('with_scale', True), ('no_scale', False)]:
    model_module.scaled_dot_product_attention = _orig_sdpa if use_scale else sdpa_no_scale

    run = wandb.init(project='da6401-a3', name=f'exp2-{exp_name}',
                     config={**base_cfg, 'scaling': use_scale})
    m = build_model(base_cfg, len(train_ds.src_vocab), len(train_ds.tgt_vocab), PAD_IDX, device)
    opt = torch.optim.Adam(m.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
    sched = NoamScheduler(opt, d_model=base_cfg['d_model'], warmup_steps=4000)
    lf = LabelSmoothingLoss(len(train_ds.tgt_vocab), PAD_IDX, 0.1)

    grad_steps = 0
    m.train()
    for src, tgt in train_loader:
        if grad_steps >= 1000:
            break
        src, tgt = src.to(device), tgt.to(device)
        sm = make_src_mask(src)
        ti, to_ = tgt[:, :-1], tgt[:, 1:]
        tm = make_tgt_mask(ti)
        logits = m(src, ti, sm, tm)
        B, T, V = logits.shape
        loss = lf(logits.reshape(B*T, V), to_.reshape(-1))
        opt.zero_grad()
        loss.backward()
        q_grad = m.encoder.layers[0].self_attn.W_q.weight.grad
        k_grad = m.encoder.layers[0].self_attn.W_k.weight.grad
        q_norm = q_grad.norm().item() if q_grad is not None else 0.0
        k_norm = k_grad.norm().item() if k_grad is not None else 0.0
        wandb.log({'q_grad_norm': q_norm, 'k_grad_norm': k_norm,
                   'train_loss': loss.item(), 'step': grad_steps})
        nn.utils.clip_grad_norm_(m.parameters(), 1.0)
        opt.step()
        sched.step()
        grad_steps += 1

    run.finish()
    print(f'{exp_name} done')

model_module.scaled_dot_product_attention = _orig_sdpa

### Experiment 3: Attention Head Visualization


In [ ]:
import matplotlib.pyplot as plt

ckpt = torch.load('/kaggle/working/best_checkpoint.pt', map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

sample_de = 'Ein Hund rennt durch das Gras.'
de_toks = ['<sos>'] + [t.text.lower() for t in train_ds.spacy_de.tokenizer(sample_de)] + ['<eos>']
src_ids = [train_ds.src_vocab.get(t, train_ds.src_vocab['<unk>']) for t in de_toks]
src_t   = torch.tensor([src_ids], dtype=torch.long).to(device)
src_m   = make_src_mask(src_t)

attn_store = []

def hook_fn(module, inp, out):
    with torch.no_grad():
        q, k = inp[0], inp[1]
        b = q.size(0)
        Q = module.W_q(q).view(b, -1, module.num_heads, module.d_k).transpose(1, 2)
        K = module.W_k(k).view(b, -1, module.num_heads, module.d_k).transpose(1, 2)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(module.d_k)
        w = torch.softmax(scores, dim=-1)
        attn_store.append(w.detach().cpu())

hook = model.encoder.layers[-1].self_attn.register_forward_hook(hook_fn)
with torch.no_grad():
    _ = model.encode(src_t, src_m)
hook.remove()

attn = attn_store[0][0]
num_heads = attn.shape[0]

run = wandb.init(project='da6401-a3', name='exp3-attn-heads')
fig, axes = plt.subplots(2, num_heads // 2, figsize=(18, 7))
axes = axes.flatten()
for h in range(num_heads):
    ax = axes[h]
    im = ax.imshow(attn[h].numpy(), cmap='Blues', vmin=0, vmax=1)
    ax.set_title(f'Head {h+1}', fontsize=9)
    ax.set_xticks(range(len(de_toks)))
    ax.set_xticklabels(de_toks, rotation=45, ha='right', fontsize=7)
    ax.set_yticks(range(len(de_toks)))
    ax.set_yticklabels(de_toks, fontsize=7)
plt.suptitle('Last Encoder Layer — Per-Head Attention Weights', fontsize=12)
plt.tight_layout()
wandb.log({'attention_heads': wandb.Image(fig)})
plt.savefig('/kaggle/working/attn_heads.png', dpi=120, bbox_inches='tight')
plt.show()
run.finish()
print('Attention heatmaps logged.')

### Experiment 4: Sinusoidal vs Learned Positional Encoding


In [ ]:
class LearnedPE(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=512):
        super().__init__()
        self.pe = nn.Embedding(max_len, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        pos = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        return self.dropout(x + self.pe(pos))

def build_model_with_pe(pe_type, cfg, src_vsz, tgt_vsz, pad_idx, device):
    m = build_model(cfg, src_vsz, tgt_vsz, pad_idx, device)
    if pe_type == 'learned':
        src_emb = m.src_embed[0]
        tgt_emb = m.tgt_embed[0]
        m.src_embed = nn.Sequential(src_emb, LearnedPE(cfg['d_model'], cfg['dropout'])).to(device)
        m.tgt_embed = nn.Sequential(tgt_emb, LearnedPE(cfg['d_model'], cfg['dropout'])).to(device)
    return m

EPOCHS_PE = 10
for pe_type in ['sinusoidal', 'learned']:
    run = wandb.init(project='da6401-a3', name=f'exp4-pe-{pe_type}',
                     config={**base_cfg, 'pe_type': pe_type})
    m = build_model_with_pe(pe_type, base_cfg,
                            len(train_ds.src_vocab), len(train_ds.tgt_vocab),
                            PAD_IDX, device)
    opt = torch.optim.Adam(m.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
    sched = NoamScheduler(opt, d_model=base_cfg['d_model'], warmup_steps=4000)
    lf = LabelSmoothingLoss(len(train_ds.tgt_vocab), PAD_IDX, 0.1)
    for ep in range(EPOCHS_PE):
        tr = run_epoch(train_loader, m, lf, opt, sched, ep, True,  device)
        vl = run_epoch(val_loader,   m, lf, None, None,   ep, False, device)
        val_bleu = evaluate_bleu(m, val_loader, tgt_vocab_obj, device, max_len=60)
        wandb.log({'train_loss': tr, 'val_loss': vl, 'val_bleu': val_bleu, 'epoch': ep})
        print(f'[{pe_type}] Epoch {ep+1}  train={tr:.4f}  val={vl:.4f}  bleu={val_bleu:.2f}')
    run.finish()
    print(f'pe_type={pe_type} done\n')

### Experiment 5: Label Smoothing (eps=0.0 vs eps=0.1)


In [ ]:
for ls_val in [0.0, 0.1]:
    run = wandb.init(project='da6401-a3', name=f'exp5-ls{int(ls_val*10)}',
                     config={**base_cfg, 'label_smoothing': ls_val})
    m = build_model(base_cfg, len(train_ds.src_vocab), len(train_ds.tgt_vocab), PAD_IDX, device)
    opt = torch.optim.Adam(m.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
    sched = NoamScheduler(opt, d_model=base_cfg['d_model'], warmup_steps=4000)
    lf = LabelSmoothingLoss(len(train_ds.tgt_vocab), PAD_IDX, ls_val)

    for ep in range(EPOCHS_ABL):
        m.train()
        for src, tgt in train_loader:
            src, tgt = src.to(device), tgt.to(device)
            sm = make_src_mask(src)
            ti, to_ = tgt[:, :-1], tgt[:, 1:]
            tm = make_tgt_mask(ti)
            logits = m(src, ti, sm, tm)
            B, T, V = logits.shape
            loss = lf(logits.reshape(B*T, V), to_.reshape(-1))
            with torch.no_grad():
                probs = torch.softmax(logits, dim=-1)
                valid_mask = (to_ != PAD_IDX)
                correct_p = probs.gather(2, to_.unsqueeze(-1)).squeeze(-1)
                conf = correct_p[valid_mask].mean().item()
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            opt.step()
            sched.step()
            wandb.log({'train_loss': loss.item(), 'pred_confidence': conf})
        vl = run_epoch(val_loader, m, lf, None, None, ep, False, device)
        wandb.log({'val_loss': vl, 'epoch': ep})
        print(f'[ls={ls_val}] Epoch {ep+1}  val={vl:.4f}')
    run.finish()
    print(f'ls={ls_val} done\n')

## Final Step: Upload files to Google Drive

1. Download `best_checkpoint.pt` and `vocabs.pt` from `/kaggle/working/`
2. Upload both to Google Drive → set sharing to **Anyone with the link → Viewer**
3. Copy each file's ID from the sharing URL
4. In `model.py`, replace `VOCAB_GDRIVE_FILE_ID` and `WEIGHTS_GDRIVE_FILE_ID` with the real IDs
5. Submit updated `model.py`, `train.py`, `lr_scheduler.py`, `dataset.py` to Gradescope
